# Week 2b — Evaluation (Verification + Identification Sweep)

Loads a trained embedding model (either the **pretrained baseline** or your **from-scratch checkpoint**) and evaluates it on:
1. **Verification (SV)** — EER / minDCF, single run.
2. **Identification (SID)** — swept across multiple `(n_enroll, n_test)` combinations, e.g. answering "if a user enrolls with 3 utterances, how accurately can the assistant identify them later?"

Because embeddings are cached after the first pass, the sweep is cheap — no re-running the model, just re-grouping and re-scoring cached embeddings.

**To evaluate the from-scratch model:** attach the Week 2a training notebook's output (containing `checkpoints/final_checkpoint.pt` and `checkpoints/model_config.json`) as an input dataset here.
**To evaluate the pretrained baseline instead:** just set `MODEL_TYPE = "pretrained"` below — no attached checkpoint needed, just Internet ON.

In [ ]:
!pip install -q speechbrain


In [ ]:
import os, random, pickle, json
from pathlib import Path
from collections import defaultdict
import torch
import torchaudio
import torch.nn.functional as F
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


In [ ]:
# ---- Config: choose which model to evaluate ----
MODEL_TYPE = "scratch"   # "scratch" or "pretrained"


In [ ]:
path_split = Path("/kaggle/input/datasets/sabahesaraki/voxceleb-1-dataset")
path_data = Path("/kaggle/input/datasets/kryakrya")
path_data_train = path_data.joinpath("voxceleb1train/wav")
path_data_test  = path_data.joinpath("voxceleb1test/wav")

VERI_TEST_PATH = path_split / "veri_test2.txt"
IDEN_SPLIT_PATH = path_split / "iden_split.txt"

def resolve_wav(rel_path: str) -> Path:
    p_test = path_data_test / rel_path
    if p_test.exists():
        return p_test
    p_train = path_data_train / rel_path
    if p_train.exists():
        return p_train
    raise FileNotFoundError(f"Could not find {rel_path}")

SAMPLE_RATE = 16000
print("Paths OK. Evaluating MODEL_TYPE =", MODEL_TYPE)


## 1. Load the model

In [ ]:
if MODEL_TYPE == "pretrained":
    from speechbrain.inference.speaker import EncoderClassifier
    classifier_model = EncoderClassifier.from_hparams(
        source="speechbrain/spkrec-ecapa-voxceleb",
        savedir="/kaggle/working/pretrained_ecapa",
        run_opts={"device": device}
    )

    @torch.no_grad()
    def extract_embedding(rel_path: str) -> torch.Tensor:
        full_path = resolve_wav(rel_path)
        signal, fs = torchaudio.load(str(full_path))
        if fs != SAMPLE_RATE:
            signal = torchaudio.functional.resample(signal, fs, SAMPLE_RATE)
        signal = signal.to(device)
        return classifier_model.encode_batch(signal).squeeze().cpu()

    print("Loaded pretrained speechbrain/spkrec-ecapa-voxceleb.")

elif MODEL_TYPE == "scratch":
    from speechbrain.lobes.models.ECAPA_TDNN import ECAPA_TDNN
    from speechbrain.lobes.features import Fbank
    from speechbrain.processing.features import InputNormalization

    # Locate the checkpoint + config from the attached Week 2a output
    ckpt_candidates = list(Path("/kaggle/input").rglob("final_checkpoint.pt"))
    cfg_candidates = list(Path("/kaggle/input").rglob("model_config.json"))
    assert ckpt_candidates, "final_checkpoint.pt not found - attach the Week 2a training notebook's output as an input dataset"
    assert cfg_candidates, "model_config.json not found - attach the Week 2a training notebook's output as an input dataset"

    ckpt_path = ckpt_candidates[0]
    cfg_path = cfg_candidates[0]
    print("Using checkpoint:", ckpt_path)
    print("Using config:    ", cfg_path)

    with open(cfg_path) as f:
        model_config = json.load(f)

    compute_features = Fbank(n_mels=model_config["n_mels"]).to(device)
    mean_var_norm = InputNormalization(norm_type="sentence", std_norm=False).to(device)

    embedding_model = ECAPA_TDNN(
        input_size=model_config["n_mels"],
        channels=model_config["channels"],
        kernel_sizes=model_config["kernel_sizes"],
        dilations=model_config["dilations"],
        groups=[1] * len(model_config["channels"]),
        attention_channels=model_config["attention_channels"],
        lin_neurons=model_config["emb_dim"],
    ).to(device)

    ckpt = torch.load(ckpt_path, map_location=device)
    embedding_model.load_state_dict(ckpt["embedding_model"])
    embedding_model.eval()
    print(f"Loaded from-scratch checkpoint (trained {ckpt['epoch']} epochs, "
          f"{model_config['n_speakers']} speakers).")

    @torch.no_grad()
    def extract_embedding(rel_path: str) -> torch.Tensor:
        full_path = resolve_wav(rel_path)
        signal, fs = torchaudio.load(str(full_path))
        if fs != SAMPLE_RATE:
            signal = torchaudio.functional.resample(signal, fs, SAMPLE_RATE)
        signal = signal.mean(dim=0, keepdim=True).to(device)
        feats = compute_features(signal)
        lens = torch.ones(feats.shape[0], device=device)
        feats = mean_var_norm(feats, lens)
        emb = embedding_model(feats).squeeze().cpu()
        return emb

else:
    raise ValueError("MODEL_TYPE must be 'pretrained' or 'scratch'")

embedding_cache = {}
def get_embedding(rel_path: str) -> torch.Tensor:
    if rel_path not in embedding_cache:
        embedding_cache[rel_path] = extract_embedding(rel_path)
    return embedding_cache[rel_path]

# Warm-up check
test_line = open(VERI_TEST_PATH).readline().split()
_ = get_embedding(test_line[1])
print("Embedding shape:", _.shape)


## 2. Verification (EER / minDCF) — single run

In [ ]:
trials = []
with open(VERI_TEST_PATH) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) != 3:
            continue
        label, p1, p2 = parts
        trials.append((int(label), p1, p2))

N_TRIALS = 3000  # set None for the full trial set
random.seed(42)
eval_trials = trials if N_TRIALS is None else random.sample(trials, N_TRIALS)

unique_files = set()
for _, p1, p2 in eval_trials:
    unique_files.update([p1, p2])
for f in tqdm(unique_files, desc="Embedding SV test files"):
    get_embedding(f)

scores, labels = [], []
for label, p1, p2 in eval_trials:
    e1, e2 = get_embedding(p1), get_embedding(p2)
    sim = F.cosine_similarity(e1.unsqueeze(0), e2.unsqueeze(0)).item()
    scores.append(sim)
    labels.append(label)
scores, labels = np.array(scores), np.array(labels)

from sklearn.metrics import roc_curve

def compute_eer(labels, scores):
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    return (fpr[idx] + fnr[idx]) / 2

def compute_min_dcf(labels, scores, p_target=0.05, c_miss=1, c_fa=1):
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    dcf = c_miss * fnr * p_target + c_fa * fpr * (1 - p_target)
    i = np.argmin(dcf)
    norm = min(c_miss * p_target, c_fa * (1 - p_target))
    return dcf[i] / norm

sv_eer = compute_eer(labels, scores)
sv_min_dcf = compute_min_dcf(labels, scores)
print(f"SV EER:    {sv_eer*100:.2f}%")
print(f"SV minDCF: {sv_min_dcf:.4f}")


## 3. Identification — enrollment/test size sweep

This is the experiment you wanted: for each `(n_enroll, n_test)` combination, build per-speaker centroids from `n_enroll` enrollment utterances and classify `n_test` held-out utterances per speaker, repeated across all speakers in the official `iden_split.txt` test partition. All embeddings are cached, so only the centroid-building and nearest-neighbor comparison steps get redone per sweep point — fast.

In [ ]:
iden_entries = []
with open(IDEN_SPLIT_PATH) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) != 2:
            continue
        split_label, rel_path = parts
        speaker_id = rel_path.split('/')[0]
        iden_entries.append((int(split_label), rel_path, speaker_id))

train_entries = [e for e in iden_entries if e[0] == 1]
test_entries  = [e for e in iden_entries if e[0] == 3]

by_speaker_train = defaultdict(list)
for split, rel_path, spk in train_entries:
    by_speaker_train[spk].append(rel_path)
by_speaker_test = defaultdict(list)
for split, rel_path, spk in test_entries:
    by_speaker_test[spk].append(rel_path)

print("Speakers with train utterances:", len(by_speaker_train))
print("Speakers with test utterances: ", len(by_speaker_test))
print("Max utterances available per speaker (train):", max(len(v) for v in by_speaker_train.values()))
print("Max utterances available per speaker (test): ", max(len(v) for v in by_speaker_test.values()))


In [ ]:
def evaluate_identification(n_enroll, n_test, seed=42):
    """Build per-speaker centroids from n_enroll utterances, classify n_test held-out
    utterances per speaker via nearest centroid (cosine). Returns top1/top5 acc + n evaluated."""
    rng = random.Random(seed)

    enroll_set = {}
    for spk, files in by_speaker_train.items():
        if len(files) >= n_enroll:
            enroll_set[spk] = rng.sample(files, n_enroll)

    eval_set = {}
    for spk, files in by_speaker_test.items():
        if spk in enroll_set and len(files) >= n_test:
            eval_set[spk] = rng.sample(files, n_test)

    speaker_centroids = {}
    for spk, files in enroll_set.items():
        embs = torch.stack([get_embedding(f) for f in files])
        speaker_centroids[spk] = F.normalize(embs.mean(dim=0), dim=0)

    centroid_ids = list(speaker_centroids.keys())
    centroid_matrix = torch.stack([speaker_centroids[s] for s in centroid_ids])

    correct, top5_correct, total = 0, 0, 0
    for spk, files in eval_set.items():
        for f in files:
            emb = F.normalize(get_embedding(f), dim=0)
            sims = centroid_matrix @ emb
            pred_spk = centroid_ids[torch.argmax(sims).item()]
            top5_spk = [centroid_ids[i] for i in torch.topk(sims, k=min(5, len(centroid_ids))).indices.tolist()]
            total += 1
            correct += int(pred_spk == spk)
            top5_correct += int(spk in top5_spk)

    return {
        'n_enroll': n_enroll,
        'n_test': n_test,
        'n_speakers': len(eval_set),
        'n_utterances_evaluated': total,
        'top1_acc': correct / total if total else float('nan'),
        'top5_acc': top5_correct / total if total else float('nan'),
    }

# Quick sanity check with a single config
print(evaluate_identification(n_enroll=3, n_test=3))


In [ ]:
# ---- The sweep ----
# Primary question: how does enrollment size affect identification accuracy?
ENROLL_OPTIONS = [1, 2, 3, 5, 10]
TEST_OPTIONS = [1, 3, 5]

sweep_results = []
for n_test in TEST_OPTIONS:
    for n_enroll in ENROLL_OPTIONS:
        res = evaluate_identification(n_enroll=n_enroll, n_test=n_test)
        sweep_results.append(res)
        print(f"enroll={n_enroll:>2}  test={n_test:>2}  "
              f"top1={res['top1_acc']*100:5.1f}%  top5={res['top5_acc']*100:5.1f}%  "
              f"(n_speakers={res['n_speakers']}, n_utt={res['n_utterances_evaluated']})")

sweep_df = pd.DataFrame(sweep_results)
sweep_df.to_csv('/kaggle/working/identification_sweep_results.csv', index=False)
sweep_df


## 4. Plot: accuracy vs. enrollment size

In [ ]:
plt.figure(figsize=(8,5))
for n_test in TEST_OPTIONS:
    subset = sweep_df[sweep_df['n_test'] == n_test].sort_values('n_enroll')
    plt.plot(subset['n_enroll'], subset['top1_acc'] * 100, marker='o', label=f'{n_test} test utterance(s)')

plt.xlabel('Number of enrollment utterances')
plt.ylabel('Top-1 identification accuracy (%)')
plt.title(f'Identification accuracy vs. enrollment size ({MODEL_TYPE} model)')
plt.legend(title='Evaluated on:')
plt.grid(alpha=0.3)
plt.savefig('/kaggle/working/identification_sweep_plot.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Heatmap view (both dimensions at once)

In [ ]:
pivot = sweep_df.pivot(index='n_test', columns='n_enroll', values='top1_acc') * 100

plt.figure(figsize=(7,4))
plt.imshow(pivot.values, cmap='viridis', aspect='auto')
plt.colorbar(label='Top-1 accuracy (%)')
plt.xticks(range(len(pivot.columns)), pivot.columns)
plt.yticks(range(len(pivot.index)), pivot.index)
plt.xlabel('n_enroll'); plt.ylabel('n_test')
plt.title(f'Top-1 identification accuracy grid ({MODEL_TYPE} model)')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        plt.text(j, i, f'{pivot.values[i,j]:.0f}', ha='center', va='center', color='white')
plt.tight_layout()
plt.savefig('/kaggle/working/identification_sweep_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ---- Save everything ----
results = {
    'model_type': MODEL_TYPE,
    'sv': {'eer': float(sv_eer), 'min_dcf': float(sv_min_dcf), 'n_trials': len(eval_trials)},
    'sid_sweep': sweep_results,
}
with open(f'/kaggle/working/eval_results_{MODEL_TYPE}.pkl', 'wb') as f:
    pickle.dump(results, f)

print(f"Saved eval_results_{MODEL_TYPE}.pkl")
print(f"\n=== SUMMARY ===")
print(f"Model: {MODEL_TYPE}")
print(f"SV EER: {sv_eer*100:.2f}%  |  minDCF: {sv_min_dcf:.4f}")
print(f"SID accuracy range across sweep: {sweep_df['top1_acc'].min()*100:.1f}% - {sweep_df['top1_acc'].max()*100:.1f}%")


## Report-writing notes

- The enrollment-size sweep plot directly answers "how many utterances does a user need to enroll with for reliable recognition?" — a natural design decision to justify in your report's enrollment procedure section.
- Run this notebook twice — once with `MODEL_TYPE = "pretrained"`, once with `MODEL_TYPE = "scratch"` — to get the enrollment-size sensitivity for both, and overlay them in the report if useful.
- The `n_speakers` column in the sweep table matters: with very high `n_enroll` values, fewer speakers will have enough train utterances to qualify, which can subtly skew the reported accuracy (evaluated on an easier/smaller subset of speakers). Worth a one-line caveat in the report if `n_speakers` drops noticeably at the high end of your sweep.
